In [1]:
!pip install duckdb -q

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
OCT_PATH = '/content/drive/MyDrive/assignment/archive/2019-Oct.csv'
NOV_PATH = '/content/drive/MyDrive/assignment/archive/2019-Nov.csv'

In [3]:
import duckdb

con = duckdb.connect()

# Column names and types
con.execute(f"DESCRIBE SELECT * FROM read_csv_auto('{OCT_PATH}')").df()

,column_name,column_type,null,key,default,extra
0,event_time,TIMESTAMP,YES,None,None,None
1,event_type,VARCHAR,YES,None,None,None
2,product_id,BIGINT,YES,None,None,None
3,category_id,BIGINT,YES,None,None,None
4,category_code,VARCHAR,YES,None,None,None
5,brand,VARCHAR,YES,None,None,None
6,price,DOUBLE,YES,None,None,None
7,user_id,BIGINT,YES,None,None,None
8,user_session,VARCHAR,YES,None,None,None


In [4]:
con.execute(f"""
    SELECT
        COUNT(*)                          AS total_rows,
        MIN(event_time)                   AS earliest,
        MAX(event_time)                   AS latest,
        COUNT(DISTINCT user_id)           AS unique_users,
        COUNT(DISTINCT product_id)        AS unique_products,
        COUNT(DISTINCT user_session)      AS unique_sessions
    FROM read_csv_auto('{OCT_PATH}')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,earliest,latest,unique_users,unique_products,unique_sessions
0,42448764,2019-10-01,2019-10-31 23:59:59,3022290,166794,9244421


In [5]:
con.execute(f"""
    SELECT event_type, COUNT(*) AS cnt,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM read_csv_auto('{OCT_PATH}')
    GROUP BY event_type
    ORDER BY cnt DESC
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,event_type,cnt,pct
0,view,40779399,96.07
1,cart,926516,2.18
2,purchase,742849,1.75


In [6]:
con.execute(f"""
    SELECT
        COUNT(*) FILTER (WHERE event_time   IS NULL) AS null_event_time,
        COUNT(*) FILTER (WHERE event_type   IS NULL) AS null_event_type,
        COUNT(*) FILTER (WHERE product_id   IS NULL) AS null_product_id,
        COUNT(*) FILTER (WHERE category_id  IS NULL) AS null_category_id,
        COUNT(*) FILTER (WHERE category_code IS NULL) AS null_category_code,
        COUNT(*) FILTER (WHERE brand        IS NULL) AS null_brand,
        COUNT(*) FILTER (WHERE price        IS NULL) AS null_price,
        COUNT(*) FILTER (WHERE user_id      IS NULL) AS null_user_id,
        COUNT(*) FILTER (WHERE user_session IS NULL) AS null_user_session
    FROM read_csv_auto('{OCT_PATH}')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,null_event_time,null_event_type,null_product_id,null_category_id,null_category_code,null_brand,null_price,null_user_id,null_user_session
0,0,0,0,0,13515609,6113008,0,0,2


In [7]:
con.execute(f"""
    SELECT
        MIN(price)  AS min_price,
        MAX(price)  AS max_price,
        AVG(price)  AS avg_price,
        COUNT(*) FILTER (WHERE price <= 0)      AS zero_or_negative,
        COUNT(*) FILTER (WHERE price > 10000)   AS suspiciously_large
    FROM read_csv_auto('{OCT_PATH}')
    WHERE event_type = 'purchase'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_price,max_price,avg_price,zero_or_negative,suspiciously_large
0,0.77,2574.07,309.561569,0,0


In [8]:
con.execute(f"""
    SELECT COUNT(*) AS duplicate_events FROM (
        SELECT user_id, product_id, event_time, COUNT(*) AS c
        FROM read_csv_auto('{OCT_PATH}')
        GROUP BY user_id, product_id, event_time
        HAVING c > 1
    )
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_events
0,28081


In [10]:
with open('/content/drive/MyDrive/assignment/schema/ddl.sql', 'r') as f:
    sql = f.read()

for stmt in sql.split(';'):
    stmt = stmt.strip()
    if stmt:
        con.execute(stmt)

print("Tables created:")
print(con.execute("SHOW TABLES").df())

Tables created:
           name
0  dim_category
1   dim_product
2      dim_user
3   fact_events


In [11]:
# Validation query 1: every product in fact_events exists in dim_product
orphan_products = con.execute("""
    SELECT COUNT(*) AS orphan_product_refs
    FROM fact_events f
    LEFT JOIN dim_product p ON f.product_id = p.product_id
    WHERE p.product_id IS NULL
""").fetchone()[0]
print(f"Orphan product references: {orphan_products}  ✓" if orphan_products == 0 else f"⚠ {orphan_products} orphans found")

# Validation query 2: every user in fact_events exists in dim_user
orphan_users = con.execute("""
    SELECT COUNT(*) AS orphan_user_refs
    FROM fact_events f
    LEFT JOIN dim_user u ON f.user_id = u.user_id
    WHERE u.user_id IS NULL
""").fetchone()[0]
print(f"Orphan user references: {orphan_users}  ✓" if orphan_users == 0 else f"⚠ {orphan_users} orphans found")

# Validation query 3: no NULLs in NOT NULL columns
nulls = con.execute("""
    SELECT
        COUNT(*) FILTER (WHERE event_time   IS NULL) AS null_event_time,
        COUNT(*) FILTER (WHERE event_type   IS NULL) AS null_event_type,
        COUNT(*) FILTER (WHERE product_id   IS NULL) AS null_product_id,
        COUNT(*) FILTER (WHERE user_id      IS NULL) AS null_user_id,
        COUNT(*) FILTER (WHERE user_session IS NULL) AS null_user_session
    FROM fact_events
""").df()
print(nulls)

Orphan product references: 0  ✓
Orphan user references: 0  ✓
   null_event_time  null_event_type  null_product_id  null_user_id  \
0                0                0                0             0   

   null_user_session  
0                  0  


In [12]:
con.close()